# Smart Academic Email Generator using NLP
### NLP Lab Mini Project

**Course:** Natural Language Processing Lab
**Program:** B.Tech / B.E. — Artificial Intelligence & Data Science


## 1. Introduction

Writing a formal academic email (leave application, permission request, apology
for absence, etc.) is a routine but often time-consuming task for students.
Students frequently know *why* they need to write the email (a few keywords)
but struggle to phrase it formally.

This project builds a **Smart Academic Email Generator** that converts a
handful of informal keywords into a complete, grammatically correct,
professional email — using **classical Natural Language Processing (NLP)
techniques only**. No OpenAI/Claude/Gemini API or any Large Language Model
is used for generating the email content; every stage below is a rule-based
or classical statistical NLP/ML technique that can be fully explained and
justified in a viva.


## 2. Problem Statement

Given short, informal, and possibly incomplete keyword input from a student
(e.g. *"hospital, mother admitted, urgent, missed practical, sorry"*), along
with basic details (teacher name, student name, roll number, department,
date), automatically:

1. Understand the **intent** of the email (medical leave, apology, permission
   request, etc.)
2. Extract relevant **entities** (dates, durations, people mentioned)
3. Determine the **urgency** of the situation
4. Generate a complete, grammatically well-formed **subject line and email
   body** using template-based Natural Language Generation (NLG)

—all without depending on any generative language model.


## 3. Objectives

- Build a complete classical NLP pipeline: cleaning → tokenization →
  stopword removal → lemmatization → POS tagging → keyword extraction →
  NER → intent classification → reason classification.
- Support **15 different academic email intents** with distinct templates.
- Provide an explainable **confidence score** for the predicted intent using
  TF-IDF + cosine similarity (scikit-learn), combined with a transparent
  rule-based keyword-lexicon vote.
- Build an interactive **Streamlit** web application with export (TXT/PDF),
  editing, and dark-mode support.
- Keep every step explainable for a viva — no "black box" LLM generation.


## 4. Theory / NLP Concepts Used

| Stage | Technique | Why it is used |
|---|---|---|
| Text Cleaning | Regex normalisation | Removes noisy punctuation, normalises case |
| Tokenization | NLTK `word_tokenize` (regex fallback) | Splits raw text into word units |
| Stopword Removal | NLTK stopword corpus | Removes low-information function words |
| Lemmatization | WordNet Lemmatizer / spaCy | Normalises word inflections to base form |
| POS Tagging | spaCy statistical tagger / NLTK | Filters keywords to content-bearing POS classes (NOUN, VERB, ADJ, PROPN) |
| Keyword Extraction | Term-frequency ranking + POS filter | Surfaces the most informative words/phrases |
| Named Entity Recognition | spaCy pretrained NER model (`en_core_web_sm`) + regex fallback | Extracts DATE, PERSON, ORG, GPE, DURATION |
| Intent Classification | Hand-built keyword lexicon voting **+** TF-IDF vectorisation with cosine similarity (scikit-learn) | Combines an explainable rule-based signal with a classical ML similarity signal |
| Urgency Detection | Lexicon-based rule matching | Flags emergencies for urgent phrasing |
| Reason Classification | Rule-based phrase construction from keywords + entities | Converts raw keywords into a natural reason clause |
| Template Selection | Dictionary lookup keyed by predicted intent | Chooses the correct email template family |
| NLG (email generation) | Slot-filling / template-based generation | Deterministic, explainable text generation — **not** a language model |

All of the above are **classical / statistical NLP techniques**. spaCy's NER
model, for example, is a transition-based / CRF-style sequence tagger
trained for entity recognition — it is *not* a generative large language
model, and it plays no role in composing the email text itself.


## 5. Dataset

No external labelled dataset is required. The project instead uses:

- A **hand-curated keyword lexicon** per intent (15 intents) for rule-based
  voting (see `intent_classifier.py::INTENT_LEXICON`).
- A small set of **representative example sentences** per intent, used only
  to build the TF-IDF similarity space for the confidence score — this acts
  as a tiny reference corpus rather than a training set (no model
  parameters are learned/fitted and saved; TF-IDF is fit fresh on each
  request against the fixed example corpus).

This keeps the system fully deterministic, transparent, and easy to extend
(a new intent can be added by writing a new lexicon + example sentences +
template — no retraining required).


## 6. Complete Implementation

### 6.1 Environment Setup

Install dependencies (already listed in `requirements.txt`):

```bash
pip install streamlit spacy nltk scikit-learn pandas reportlab
python -m spacy download en_core_web_sm
```

The cells below import the project's own modules (`nlp_utils.py`,
`intent_classifier.py`, `templates.py`, `email_generator.py`) so that the
notebook and the Streamlit app share **exactly the same pipeline code**.


In [1]:
# Make sure we can import the project modules from this notebook's folder
import sys, os
sys.path.insert(0, os.getcwd())

import nlp_utils as nu
import intent_classifier as ic
import templates as tpl
from email_generator import generate_email

print("spaCy model loaded:", nu.SPACY_AVAILABLE)
print("NLTK corpora available:", nu.NLTK_AVAILABLE)


spaCy model loaded: True
NLTK corpora available: True


### 6.2 Stage 1 — Text Cleaning

In [2]:
sample_input = '''cousin accident
hospital
father in office
urgent
missed lab
21 September'''

cleaned = nu.clean_text(sample_input)
print("RAW INPUT:\n", sample_input)
print("\nCLEANED TEXT:\n", cleaned)


RAW INPUT:
 cousin accident
hospital
father in office
urgent
missed lab
21 September

CLEANED TEXT:
 cousin accident hospital father in office urgent missed lab 21 september


### 6.3 Stage 2 — Tokenization

In [3]:
tokens = nu.tokenize(cleaned)
print(tokens)


['cousin', 'accident', 'hospital', 'father', 'in', 'office', 'urgent', 'missed', 'lab', '21', 'september']


### 6.4 Stage 3 — Stopword Removal

In [4]:
tokens_ns = nu.remove_stopwords(tokens)
print(tokens_ns)


['cousin', 'accident', 'hospital', 'father', 'office', 'urgent', 'missed', 'lab', '21', 'september']


### 6.5 Stage 4 — Lemmatization

In [5]:
lemmas = nu.lemmatize(tokens_ns)
print(lemmas)


['cousin', 'accident', 'hospital', 'father', 'office', 'urgent', 'miss', 'lab', '21', 'september']


### 6.6 Stage 5 — POS Tagging (used internally for keyword filtering)

In [6]:
pos_tags = nu.pos_tag_tokens(cleaned)
print(pos_tags)


[('cousin', 'PROPN'), ('accident', 'NOUN'), ('hospital', 'PROPN'), ('father', 'NOUN'), ('in', 'ADP'), ('office', 'NOUN'), ('urgent', 'ADJ'), ('missed', 'VERB'), ('lab', 'NOUN'), ('21', 'NUM'), ('september', 'PROPN')]


### 6.7 Stage 6 — Keyword Extraction

In [7]:
keywords = nu.extract_keywords(sample_input)
print("Extracted keywords / keyphrases:")
print(keywords)


Extracted keywords / keyphrases:
['cousin', 'accident', 'hospital', 'father', 'office', 'urgent', 'miss', 'lab', 'september', 'cousin accident', 'father in office', 'missed lab']


### 6.8 Stage 7 — Intent Classification

Two signals are combined:
1. **Lexicon vote** — overlap between extracted keywords and each intent's
   hand-curated keyword set.
2. **TF-IDF + cosine similarity** — scikit-learn `TfidfVectorizer` compares
   the raw input against representative example sentences for every intent.


In [8]:
intent_result = ic.classify_intent(sample_input, keywords)
print("Predicted intent:", intent_result['intent_label'])
print("Confidence score:", intent_result['confidence'], "%")
print()
print("Lexicon vote scores:")
for k, v in sorted(intent_result['lexicon_scores'].items(), key=lambda x: -x[1])[:5]:
    print(f"  {k:30s} {v}")
print()
print("TF-IDF cosine similarity scores:")
for k, v in sorted(intent_result['tfidf_scores'].items(), key=lambda x: -x[1])[:5]:
    print(f"  {k:30s} {v:.3f}")


Predicted intent: Family Emergency
Confidence score: 69.5 %

Lexicon vote scores:
  family_emergency               4.0
  medical_leave                  3.5
  apology_absence                2.5
  club_activity                  0.0
  technical_competition          0.0

TF-IDF cosine similarity scores:
  family_emergency               0.237
  apology_absence                0.088
  medical_leave                  0.063
  club_activity                  0.000
  technical_competition          0.000


### 6.9 Stage 8 — Named Entity Recognition

In [9]:
entities = nu.extract_entities(sample_input)
print(entities)


{'DATE': ['21 September'], 'PERSON': [], 'ORG': [], 'GPE': [], 'DURATION': []}


### 6.10 Urgency Detection

In [10]:
urgency = nu.detect_urgency(sample_input)
print("Urgency level:", urgency)


Urgency level: High


### 6.11 Stage 9 — Reason Classification, Template Selection & Email Generation

In [11]:
result = generate_email(
    teacher_name="Dr. Patil",
    student_name="Paras Nalte",
    keywords_text=sample_input,
    roll_number="21",
    department="AI & Data Science",
    date="21 September 2025",
)

print("SUBJECT:", result['subject'])
print()
print(result['body'])


SUBJECT: Leave Application Due to Family Emergency on 21 September 2025

Respected Dr. Patil,

I am writing to inform you that my cousin was hospitalized and admitted for treatment. This is an urgent matter and immediate attention would be appreciated. Because of this family emergency, I was unable to attend college on 21 September 2025. I request you to kindly grant me leave and excuse my absence during this period. I will complete any pending work as soon as I am able to.

Thank you for your consideration.

Regards,
Paras Nalte
Roll No: 21
Department: AI & Data Science


### 6.12 End-to-End Pipeline Summary

The dictionary returned by `generate_email()` contains every intermediate
artefact produced by the pipeline, which is exactly what the Streamlit app's
**"NLP Pipeline" tab** displays for explainability during the viva.


In [12]:
for key in ["cleaned_text", "tokens", "tokens_no_stopwords", "lemmas",
            "keywords", "entities", "urgency", "intent_label",
            "confidence", "reason_phrase", "subject"]:
    print(f"{key:22s}: {result[key]}")


cleaned_text          : cousin accident hospital father in office urgent missed lab 21 september
tokens                : ['cousin', 'accident', 'hospital', 'father', 'in', 'office', 'urgent', 'missed', 'lab', '21', 'september']
tokens_no_stopwords   : ['cousin', 'accident', 'hospital', 'father', 'office', 'urgent', 'missed', 'lab', '21', 'september']
lemmas                : ['cousin', 'accident', 'hospital', 'father', 'office', 'urgent', 'miss', 'lab', '21', 'september']
keywords              : ['cousin', 'accident', 'hospital', 'father', 'office', 'urgent', 'miss', 'lab', 'september', 'cousin accident', 'father in office', 'missed lab']
entities              : {'DATE': ['21 September'], 'PERSON': [], 'ORG': [], 'GPE': [], 'DURATION': []}
urgency               : High
intent_label          : Family Emergency
confidence            : 69.5
reason_phrase         : my cousin was hospitalized and admitted for treatment
subject               : Leave Application Due to Family Emergency on 21 Se

### 6.13 A Second Worked Example (different intent)

Demonstrates that the same pipeline correctly switches templates for a
completely different situation — a hackathon permission request.


In [13]:
sample_input_2 = '''hackathon
selected
national level
permission
2 days'''

result2 = generate_email(
    teacher_name="Prof. Sharma",
    student_name="Riya Shah",
    keywords_text=sample_input_2,
)

print("Detected intent:", result2['intent_label'], "| confidence:", result2['confidence'], "%")
print()
print("SUBJECT:", result2['subject'])
print()
print(result2['body'])


Detected intent: Hackathon | confidence: 90.7 %

SUBJECT: Permission Request to Participate in Hackathon on 2 days

Respected Prof. Sharma,

I am excited to share that my team has been selected to participate in a hackathon. I request your permission to attend this hackathon on 2 days, as it is a great opportunity to apply my technical skills and gain practical exposure. I will complete any academic work I miss as soon as possible.

Thank you for your support and encouragement.

Regards,
Riya Shah


### 6.14 Output Screenshots

*(When running this notebook locally, launch the Streamlit app with
`streamlit run app.py`, generate an email, and paste screenshots of the
"Generated Email" and "NLP Pipeline" tabs here for your submission.)*

```
[ Insert screenshot: Generated Email tab ]
[ Insert screenshot: NLP Pipeline tab — showing tokens, keywords, entities ]
[ Insert screenshot: Edit Email tab ]
```


## 7. Advantages

- **Fully explainable**: every output can be traced back to a rule, a
  lexicon entry, or a classical statistical computation — ideal for a
  university viva.
- **No API cost / no internet dependency at runtime**: once the spaCy model
  is downloaded, the system works fully offline, unlike LLM-API-based
  generators.
- **Fast and lightweight**: no GPU or large model inference required.
- **Easily extensible**: new intents can be added by writing a new lexicon
  + example sentences + template.
- **Privacy-friendly**: no student data is sent to any third-party AI
  service.


## 8. Limitations

- Grammar correction is limited to simple rule-based capitalisation/
  punctuation fixes — it cannot fix complex grammatical errors the way a
  full grammar-checking model could.
- Reason-phrase construction is rule-based and may not perfectly capture
  very unusual or highly novel situations outside the anticipated intents.
- Intent classification accuracy depends on the quality/coverage of the
  hand-curated lexicons and example sentences; very sparse or ambiguous
  keyword input may reduce confidence.
- Named Entity Recognition quality depends on spaCy's pretrained
  `en_core_web_sm` model, which can occasionally miss informal date formats
  (mitigated here with a regex fallback, but not perfect).


## 9. Future Scope

- Add a proper rule-based / statistical grammar checker (e.g. LanguageTool
  integration) for more robust polishing.
- Train a small supervised classifier (e.g. Naive Bayes / SVM on
  TF-IDF features) on a larger, real dataset of student emails to replace
  the hand-curated lexicon with a learned model, while still avoiding LLMs.
- Support multilingual input (e.g. Hindi/Marathi keywords) using
  transliteration + classical multilingual NLP tools.
- Add email-sending integration (SMTP) so the generated email can be sent
  directly to the teacher from the app.
- Maintain a history/log of previously generated emails per student.


## 10. Conclusion

This project demonstrates a complete, explainable, end-to-end classical NLP
pipeline — text cleaning, tokenization, stopword removal, lemmatization,
POS tagging, keyword extraction, Named Entity Recognition, lexicon- and
TF-IDF-based intent classification, rule-based reason classification, and
template-based Natural Language Generation — that converts a student's raw
keywords into a complete, professional academic email, entirely **without**
relying on any Large Language Model or generative AI API. The accompanying
Streamlit application makes the tool usable end-to-end, while exposing
every pipeline stage for transparency and viva explainability.
